# Crash-to-Road Matching (Boston)

This notebook connects the crash data to the road data. The crash data has one row per crash with a
location point. The road data has one row per road segment with a line shape. Nothing links them, so
we have to figure out which segment each crash happened on.

The result is our label: for each road segment, did it have at least one non-intersection crash in
2023–2025?

**Inputs:**
- `data/processed/crashes_clean.csv` from `crash_data_cleaning.ipynb`
- `data/raw/boston_roads_2022_raw_*.json`, the Boston road download from `notebooks/01_road_inventory_cleaning.ipynb`

**Outputs (in `data/processed/`):**
- `crash_matches.csv`: one row per Boston crash, with the segment it matched to (or why it didn't match)
- `segment_labels.csv`: one row per road segment, with the label (`had_crash`)
- `matching_review_sample.csv`: 50 crashes for us to check by hand

In [ ]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import cKDTree

# Find the project folder so this works from the repo root or from notebooks/
cwd = Path.cwd()
project_root = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
RAW_DIR = project_root / "data" / "raw"
OUT_DIR = project_root / "data" / "processed"

# Matching settings (explained in section 3)
MAX_DIST = 10   # meters: a crash farther than this from every road doesn't match
TIE_DIST = 1    # meters: segments within this distance of the closest one count as a tie

## 1. Load the data

We only use Boston crashes because the road data is only Boston for now (`City = 35`). If we add more
towns later, we just need to download their roads and change the town filter.

We use the road file's `GlobalID` as `segment_id`, the same ID Weitung's cleaning notebook uses, so
our labels can be joined to their cleaned road table.

In [ ]:
road_files = sorted(RAW_DIR.glob("boston_roads_2022_raw_*.json"))
road_path = road_files[-1]   # newest download
raw_roads = json.loads(road_path.read_text())
features = raw_roads["features"]
print("Road file:", road_path.name)
print("Coordinate system:", raw_roads["spatialReference"])  # 26986 = Mass. State Plane, meters, same as crash X/Y

roads = pd.DataFrame(f["attributes"] for f in features)
roads["segment_id"] = roads["GlobalID"]
print(f"Road segments: {len(roads):,}")

crashes = pd.read_csv(OUT_DIR / "crashes_clean.csv")
crashes = crashes[crashes["CITY_TOWN_NAME"].str.upper().eq("BOSTON")].reset_index(drop=True)
print(f"Boston crashes (clean, non-intersection): {len(crashes):,}")
crashes.groupby("YEAR").size().rename("crashes").to_frame()

Boston has a lot fewer crashes in 2024 and 2025 than in 2023. Statewide counts were about the same
every year, so this is a Boston thing. It could be how Boston police reported crashes after the new
crash form in 2024. We should keep it in mind, but it doesn't change how matching works.

## 2. Find the distance from each crash to nearby roads

Each road segment is a line made of points. For each crash we find every segment within `MAX_DIST`
meters and measure the exact distance from the crash to the line.

We cut every line into short pieces (20 m or less) and put the piece centers into a KD-tree. That way
we only measure distances to pieces near the crash instead of all 150k pieces in Boston.

Segments with no line shape (the 35 Weitung set aside for geometry review) can't be matched, so we
leave them out here.

In [ ]:
piece_seg, A, B = [], [], []
has_geometry = np.zeros(len(features), dtype=bool)
for i, f in enumerate(features):
    for path in (f.get("geometry") or {}).get("paths") or []:
        pts = np.asarray(path, dtype=float)[:, :2]
        if len(pts) < 2:
            continue
        has_geometry[i] = True
        for p0, p1 in zip(pts[:-1], pts[1:]):
            n = max(1, int(np.ceil(np.hypot(*(p1 - p0)) / 20)))
            t = np.linspace(0, 1, n + 1)[:, None]
            cut = p0 + (p1 - p0) * t
            A.append(cut[:-1]); B.append(cut[1:]); piece_seg += [i] * n

A, B, piece_seg = np.vstack(A), np.vstack(B), np.array(piece_seg)
tree = cKDTree((A + B) / 2)
roads["has_geometry"] = has_geometry
print(f"{len(piece_seg):,} line pieces, {(~has_geometry).sum()} segments with no geometry")

In [ ]:
P = crashes[["X", "Y"]].to_numpy()
nearby = tree.query_ball_point(P, MAX_DIST + 10)  # +10 because a piece's center can be up to 10 m from its end

rows = []
for k, idx in enumerate(nearby):
    if not idx:
        continue
    idx = np.asarray(idx)
    a, ab = A[idx], B[idx] - A[idx]
    length2 = np.maximum((ab ** 2).sum(axis=1), 1e-9)
    t = np.clip(((P[k] - a) * ab).sum(axis=1) / length2, 0, 1)
    dist = np.hypot(*(a + ab * t[:, None] - P[k]).T)
    per_segment = pd.Series(dist).groupby(piece_seg[idx]).min()
    for seg, d in per_segment[per_segment <= MAX_DIST].items():
        rows.append((k, seg, d))

candidates = pd.DataFrame(rows, columns=["crash_row", "seg_row", "dist"])
nearest = candidates.groupby("crash_row")["dist"].min()
print(f"Crashes with a road within {MAX_DIST} m: {len(nearest):,} of {len(crashes):,}")
nearest.describe(percentiles=[.5, .9, .99]).round(2)

Almost every crash is basically sitting on a road line (the median distance is about 0 m). MassDOT
already places crash points onto the road network when they geocode them, so matching by distance is
easy. That's also why `MAX_DIST` can be small: only a handful of crashes are more than 10 m from
every road, and those are likely in places our road data doesn't cover (like airport roads).

## 3. Pick one segment for each crash

The hard part isn't distance, it's ties. Road segments are short and connect end to end, so a crash
at the spot where one segment ends and the next begins is the same distance from both.

Our rules:
1. **Too far:** no segment within `MAX_DIST` (10 m), so the crash doesn't match.
2. **Matched:** only one segment within `TIE_DIST` (1 m) of the closest one, so we take it.
3. **Matched by name:** several segments are tied, but only one has the crash's reported road name,
   so we take that one.
4. **Ambiguous:** several segments are tied and the name doesn't break the tie (usually two pieces of
   the same street). We don't guess, so the crash stays unmatched.

We use 1 m for a tie because crash locations aren't accurate to within a meter, so a crash 0.3 m from
one segment and 0 m from another could really be on either.

### Cleaning road names for comparison

Crash road names are typed by police and look like `" COLUMBUS AVE Rte N"` or
`"Regent St Rte N  / Hulbert St Rte E"`. Road names look like `"COLUMBUS AVENUE"`. Before comparing,
we uppercase them, drop the `Rte ...` part, drop house numbers, split on `/`, and spell out
abbreviations (ST → STREET, W → WEST, 3RD → THIRD, etc.).

In [ ]:
ABBR = {
    "ST": "STREET", "AVE": "AVENUE", "AV": "AVENUE", "RD": "ROAD", "BLVD": "BOULEVARD",
    "DR": "DRIVE", "PL": "PLACE", "CT": "COURT", "SQ": "SQUARE", "PKWY": "PARKWAY",
    "HWY": "HIGHWAY", "LN": "LANE", "TER": "TERRACE", "TERR": "TERRACE", "EXPY": "EXPRESSWAY",
    "CIR": "CIRCLE", "WY": "WAY", "MT": "MOUNT",
    "N": "NORTH", "S": "SOUTH", "E": "EAST", "W": "WEST",
    "1ST": "FIRST", "2ND": "SECOND", "3RD": "THIRD", "4TH": "FOURTH", "5TH": "FIFTH",
    "6TH": "SIXTH", "7TH": "SEVENTH", "8TH": "EIGHTH", "9TH": "NINTH", "10TH": "TENTH",
}

def clean_name(s):
    s = str(s).upper()
    s = re.sub(r"\bRTE\b.*$", "", s)        # drop ' Rte N' and anything after it
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    words = s.split()
    while words and words[0].isdigit():      # drop house numbers like '666 WASHINGTON ST'
        words = words[1:]
    return " ".join(ABBR.get(w, w) for w in words)

def crash_names(rdwy):
    if pd.isna(rdwy):
        return set()
    return {n for n in (clean_name(part) for part in str(rdwy).split("/")) if n}

roads["name_clean"] = roads["St_Name"].fillna("").map(clean_name)
crashes["names_clean"] = crashes["RDWY"].map(crash_names)

# Quick look at what the cleaning does
crashes[["RDWY", "names_clean"]].dropna().sample(8, random_state=0)

In [ ]:
candidates["seg_name"] = roads["name_clean"].to_numpy()[candidates["seg_row"]]
candidates["crash_names"] = crashes["names_clean"].to_numpy()[candidates["crash_row"]]
candidates["name_agrees"] = [bool(n) and n in names
                             for n, names in zip(candidates["seg_name"], candidates["crash_names"])]

# Keep only segments tied with the closest one
candidates["gap"] = candidates["dist"] - candidates.groupby("crash_row")["dist"].transform("min")
tied = candidates[candidates["gap"] <= TIE_DIST]

def decide(group):
    if len(group) == 1:
        return pd.Series({"status": "matched", "seg_row": group["seg_row"].iloc[0]})
    named = group[group["name_agrees"]]
    if len(named) == 1:
        return pd.Series({"status": "matched by name", "seg_row": named["seg_row"].iloc[0]})
    return pd.Series({"status": "ambiguous", "seg_row": -1})

decisions = tied.groupby("crash_row")[["seg_row", "name_agrees"]].apply(decide)
decisions["n_tied"] = tied.groupby("crash_row").size()
decisions["dist"] = nearest

matches = crashes[["CRASH_NUMB", "YEAR", "X", "Y", "RDWY"]].join(decisions)
matches["status"] = matches["status"].fillna("too far")
matches["seg_row"] = matches["seg_row"].fillna(-1).astype(int)
matches["n_tied"] = matches["n_tied"].fillna(0).astype(int)

ok = matches["seg_row"] >= 0
matches["segment_id"] = pd.NA
matches.loc[ok, "segment_id"] = roads["segment_id"].to_numpy()[matches.loc[ok, "seg_row"]]
matches["segment_name"] = pd.NA
matches.loc[ok, "segment_name"] = roads["St_Name"].to_numpy()[matches.loc[ok, "seg_row"]]

# Did the matched segment's name agree with the crash's reported road?
agree = candidates.set_index(["crash_row", "seg_row"])["name_agrees"]
matches["name_agrees"] = pd.NA
idx = list(zip(matches.index[ok], matches.loc[ok, "seg_row"]))
matches.loc[ok, "name_agrees"] = agree.reindex(idx).to_numpy()

summary = matches["status"].value_counts().rename("crashes").to_frame()
summary["%"] = (summary["crashes"] / len(matches) * 100).round(1)
summary

### How often does the name agree?

For matched crashes, we check whether the segment's road name agrees with the crash's reported road.
This tells us how much to trust the matches. When they don't agree, it's usually the same road under
two names (like `VFW PARKWAY` vs. `VETERANS OF FOREIGN WARS PARKWAY`, or `I93 SB` vs.
`JOHN F FITZGERALD EXPRESSWAY`), or one of the names is missing. We keep these matches but save the
flag, so we can try a stricter "name must agree" version later as a sensitivity check.

In [ ]:
m = matches[matches["seg_row"] >= 0].copy()
both_named = m["RDWY"].notna() & m["segment_name"].fillna("").str.strip().ne("")
print(f"Matched crashes: {len(m):,}")
print(f"  both have a road name: {both_named.sum():,}")
print(f"  name agrees (when both have one): {m.loc[both_named, 'name_agrees'].astype(bool).mean():.1%}")

## 4. Build the segment labels

Each segment gets `had_crash = 1` if at least one crash matched to it, and 0 otherwise. We also keep
`crash_count`, but the label is just yes/no like the proposal says.

**Ambiguous crashes:** if a crash could be on segment A or B, we don't know which one gets the 1. If
we left both as 0, one of them would be wrong. The proposal says to leave out every segment that's
tied for an ambiguous crash, so we flag them with `near_ambiguous_crash`. The model should drop these.

A 0 means no crash we could match, not proof that no crash happened.

In [ ]:
matched = matches[matches["seg_row"] >= 0]
counts = matched.groupby("seg_row").size()

amb_crash_rows = matches.index[matches["status"] == "ambiguous"]
amb_segments = set(tied.loc[tied["crash_row"].isin(amb_crash_rows), "seg_row"])

labels = roads.loc[roads["has_geometry"], ["segment_id", "OBJECTID", "St_Name", "Facility",
                                             "F_Class", "Length"]].copy()
labels["crash_count"] = counts.reindex(labels.index).fillna(0).astype(int)
labels["had_crash"] = (labels["crash_count"] > 0).astype(int)
labels["near_ambiguous_crash"] = labels.index.isin(amb_segments)

usable = labels[~labels["near_ambiguous_crash"]]
print(f"Segments with geometry:          {len(labels):,}")
print(f"Left out (near ambiguous crash): {labels['near_ambiguous_crash'].sum():,}")
print(f"Usable segments:                 {len(usable):,}")
print(f"  had_crash = 1: {usable['had_crash'].sum():,} ({usable['had_crash'].mean():.1%})")
print(f"  had_crash = 0: {(usable['had_crash'] == 0).sum():,}")

### Positive rate by road type

Bigger roads should have more crashes because they carry more traffic. If they don't, that would be a
sign something is off with the matching.

In [ ]:
F_CLASS = {0: "Local", 1: "Interstate", 2: "Principal arterial", 3: "Principal arterial (other)",
           5: "Minor arterial", 6: "Collector"}
by_class = (usable.assign(road_class=usable["F_Class"].map(F_CLASS).fillna("Unknown"))
                  .groupby("road_class")["had_crash"].agg(segments="size", with_crash="sum", rate="mean"))
by_class["rate"] = (by_class["rate"] * 100).round(1)
by_class.sort_values("rate", ascending=False)

## 5. Save, plus a sample to check by hand

The proposal says we'll check at least 50 crash matches by hand. We pull 50 crashes: 20 plain matches,
15 where the name doesn't agree, and 15 ambiguous. For each one, look at the location on a map and
fill in whether the match (or the call that it's ambiguous) looks right.

In [ ]:
matches.drop(columns=["seg_row"]).to_csv(OUT_DIR / "crash_matches.csv", index=False)
labels.to_csv(OUT_DIR / "segment_labels.csv", index=False)

m = matches[matches["seg_row"] >= 0]
review = pd.concat([
    m[m["name_agrees"] == True].sample(20, random_state=0).assign(review_group="matched, name agrees"),
    m[m["name_agrees"] == False].sample(15, random_state=0).assign(review_group="matched, name differs"),
    matches[matches["status"] == "ambiguous"].sample(15, random_state=0).assign(review_group="ambiguous"),
])
review = review.merge(crashes[["CRASH_NUMB", "LAT", "LON"]], on="CRASH_NUMB")
review["reviewer_says_correct"] = ""
review["reviewer_notes"] = ""
review.drop(columns=["seg_row"]).to_csv(OUT_DIR / "matching_review_sample.csv", index=False)

print("Saved crash_matches.csv, segment_labels.csv, matching_review_sample.csv to", OUT_DIR)

## 6. Summary

- Almost every Boston crash sits right on a road line, so finding nearby segments is easy. The hard
  part is crashes where two segments meet.
- The road name breaks some ties. The rest are marked ambiguous, and their segments are left out so we
  don't give a 0 to a segment that might have had the crash.
- `segment_labels.csv` has the label for every Boston segment and is ready to join to the road
  features for modeling.

Still to decide:
- Which segments count for modeling. Matching uses every segment so crashes don't get pushed onto the
  wrong road, but we probably want to drop things like bike/pedestrian paths (`Facility = 12`) before
  modeling.
- Whether the tie distance (1 m) is right. We could test 0.5 m and 2 m as a sensitivity check.
- Fill in `matching_review_sample.csv` by hand.